In [54]:
import pandas as pd

In [55]:
df = pd.read_csv("../data/auth_logs.csv")

In [75]:
df.head()

,login_attempts,failed_attempts,success,username_encoded,ip_encoded
0,1,0,1,0,2
1,2,1,1,4,3
2,5,5,0,5,7
3,1,0,1,1,4
4,7,7,0,0,0


In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   timestamp        12 non-null     object
 1   username         12 non-null     object
 2   ip_address       12 non-null     object
 3   login_attempts   12 non-null     int64 
 4   failed_attempts  12 non-null     int64 
 5   success          12 non-null     object
dtypes: int64(2), object(4)
memory usage: 708.0+ bytes


In [58]:
# handle missing values
df.isnull().sum()

timestamp          0
username           0
ip_address         0
login_attempts     0
failed_attempts    0
success            0
dtype: int64

In [59]:
# clean the column before mapping
df["success"] = df["success"].str.strip().str.lower()

In [60]:
# convert target label to numeric (ml model can't learn from text labels)
df["success"] = df["success"].map({"yes": 1, "no": 0})
df.head()

,timestamp,username,ip_address,login_attempts,failed_attempts,success
0,2025-01-01 09:01:12,admin,192.168.1.10,1,0,1
1,2025-01-01 09:05:43,john,192.168.1.15,2,1,1
2,2025-01-01 09:07:10,root,45.83.22.11,5,5,0
3,2025-01-01 09:10:01,alice,192.168.1.20,1,0,1
4,2025-01-01 09:15:33,admin,103.44.92.201,7,7,0


In [61]:
# drop timesatmp for new
df=df.drop(columns=['timestamp'])
df.head()

,username,ip_address,login_attempts,failed_attempts,success
0,admin,192.168.1.10,1,0,1
1,john,192.168.1.15,2,1,1
2,root,45.83.22.11,5,5,0
3,alice,192.168.1.20,1,0,1
4,admin,103.44.92.201,7,7,0


In [62]:
# encode usernames
%pip install scikit-learn
from sklearn.preprocessing import LabelEncoder

le_user = LabelEncoder()
df["username_encoded"] = le_user.fit_transform(df["username"])

df.head()

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


,username,ip_address,login_attempts,failed_attempts,success,username_encoded
0,admin,192.168.1.10,1,0,1,0
1,john,192.168.1.15,2,1,1,4
2,root,45.83.22.11,5,5,0,5
3,alice,192.168.1.20,1,0,1,1
4,admin,103.44.92.201,7,7,0,0


In [63]:
# handle ip addresses
le_ip = LabelEncoder()
df["ip_encoded"] = le_ip.fit_transform(df["ip_address"])

df.head()
le_ip = LabelEncoder()
df["ip_encoded"] = le_ip.fit_transform(df["ip_address"])

df.head()


,username,ip_address,login_attempts,failed_attempts,success,username_encoded,ip_encoded
0,admin,192.168.1.10,1,0,1,0,2
1,john,192.168.1.15,2,1,1,4,3
2,root,45.83.22.11,5,5,0,5,7
3,alice,192.168.1.20,1,0,1,1,4
4,admin,103.44.92.201,7,7,0,0,0


In [64]:
# drop original text columns
df = df.drop(columns=["username", "ip_address"])
df.head()

,login_attempts,failed_attempts,success,username_encoded,ip_encoded
0,1,0,1,0,2
1,2,1,1,4,3
2,5,5,0,5,7
3,1,0,1,1,4
4,7,7,0,0,0


In [65]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   login_attempts    12 non-null     int64
 1   failed_attempts   12 non-null     int64
 2   success           12 non-null     int64
 3   username_encoded  12 non-null     int64
 4   ip_encoded        12 non-null     int64
dtypes: int64(5)
memory usage: 612.0 bytes


In [66]:
df["success"].unique()

array([1, 0])

In [67]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   login_attempts    12 non-null     int64
 1   failed_attempts   12 non-null     int64
 2   success           12 non-null     int64
 3   username_encoded  12 non-null     int64
 4   ip_encoded        12 non-null     int64
dtypes: int64(5)
memory usage: 612.0 bytes


In [68]:
df['success'].value_counts()

success
1    7
0    5
Name: count, dtype: int64

In [73]:
df.head()

,login_attempts,failed_attempts,success,username_encoded,ip_encoded
0,1,0,1,0,2
1,2,1,1,4,3
2,5,5,0,5,7
3,1,0,1,1,4
4,7,7,0,0,0


#### training phase

In [70]:
# separate features and label
X = df.drop(columns=["success"])
y = df["success"]


In [ ]:
X.shape,y.shape


((12, 4), (12,))

In [ ]:
# test on unseen data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)


In [77]:
X_train.shape, X_test.shape


((8, 4), (4, 4))

In [79]:
# train logistic regression model(first ml model trained)
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [80]:
y_pred = model.predict(X_test)
y_pred


array([0, 1, 1, 0])

In [81]:
# evaluate model 
from sklearn.metrics import accuracy_score

accuracy_score(y_test, y_pred)


1.0

In [82]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)


array([[2, 0],
       [0, 2]])

In [83]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         2

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4

